In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

from load_data import load_datasets
from helper_functions import eval_model,display_samples
%matplotlib inline

In [ ]:
image_size=512
batch_size=16
dataset_path='130kv2/'

In [ ]:
# load our training, validation and test datasets
train_ds, val_ds, test_ds = load_datasets(dataset_path, image_size)

# display 10 samples to make sure they loaded OK
display_samples(train_ds.take(16))

# and check how many we have in each dataset
print('Train dataset: ')
print(train_ds.cardinality())
print('\nValidation dataset: ')
print(val_ds.cardinality())
print('\nTest dataset: ')
print(test_ds.cardinality())

# lastly batch them up and cache them
train_ds = train_ds.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
# add some minor augmentation, avoiding anything which might rescale or otherwise impact perturbations
# see https://keras.io/examples/vision/image_classification_from_scratch/
data_augmentation_layers = tf.keras.Sequential([
    #tf.keras.layers.Normalization(),
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.05),
])

def data_augmentation(images):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images

In [ ]:
def make_model(input_shape, num_classes):
    inputs = tf.keras.Input(shape=input_shape)

    # Entry block
    x = data_augmentation_layers(inputs)
    # x = tf.keras.layers.Rescaling(1.0 / 255)(x) # no need for this now as we're doing it in pre
    x = tf.keras.layers.Conv2D(128, 3, strides=2, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)

    previous_block_activation = x  # Set aside residual

    for size in [256, 512]:
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.SeparableConv2D(size, 3, padding="same")(x)
        x = tf.keras.layers.BatchNormalization()(x)

        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.SeparableConv2D(size, 3, padding="same")(x)
        x = tf.keras.layers.BatchNormalization()(x)

        x = tf.keras.layers.MaxPooling2D(3, strides=2, padding="same")(x)

        # Project residual
        residual = tf.keras.layers.Conv2D(size, 1, strides=2, padding="same")(
            previous_block_activation
        )
        x = tf.keras.layers.add([x, residual])  # Add back residual
        previous_block_activation = x  # Set aside next residual

    x = tf.keras.layers.SeparableConv2D(1024, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    if num_classes == 2:
        units = 1
    else:
        units = num_classes

    x = tf.keras.layers.Dropout(0.25)(x)
    # We specify activation=None so as to return logits
    outputs = tf.keras.layers.Dense(units, activation=None)(x)
    return tf.keras.Model(inputs, outputs)

model = make_model(input_shape=(image_size,image_size) + (3,), num_classes=2)
#tf.keras.utils.plot_model(model, 'models/model.png', show_shapes=True, rankdir='TB')

In [ ]:
epochs = 10

callbacks = [
    tf.keras.callbacks.ModelCheckpoint("models/save_at_{epoch}.keras"),
]
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")],
)
model_history = model.fit(
    train_ds,
    epochs=epochs,
    callbacks=callbacks,
    validation_data=val_ds,
).history

In [ ]:
#model = tf.keras.models.load_model('save_at_3.keras')
eval_results = eval_model('initial_model',model,model_history,val_ds,test_ds)
#model_history